# 📈 Módulo 04: Matplotlib

Líneas, barras, histogramas, dispersión y subplots. Convierte datos en gráficos claros.

**PandasQuest** · versión web interactiva: [https://rosalesluciano.github.io/pandas-quest/](https://rosalesluciano.github.io/pandas-quest/#/m/m04)

**Cómo funciona:**
1. Ejecuta la celda ⚙️ de abajo (carga las tablas y el corrector).
2. En cada ejercicio escribe tu código y guarda la respuesta en `resultado`.
3. Ejecuta la celda: `comprobar()` te dirá si está bien. 🎯

¿Atascado? `ver_solucion("id")` muestra la solución (¡pero inténtalo antes! 😉)

In [ ]:
#@title ⚙️ 1) Ejecuta esta celda primero (carga datos y corrector) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    try:
        return bool(u == e)
    except Exception:
        return False


def _array_compare(u, e):
    if isinstance(u, (pd.DataFrame, pd.Series)):
        u = u.to_numpy()
    try:
        ua = np.asarray(u)
    except Exception:
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if not isinstance(u, np.ndarray) and not isinstance(u, (list, tuple)):
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if ua.shape != e.shape:
        return False, f"Forma distinta: tu array tiene shape {ua.shape} y se esperaba {e.shape}."
    num = np.issubdtype(e.dtype, np.number) or e.dtype == bool
    try:
        if num and (np.issubdtype(ua.dtype, np.number) or ua.dtype == bool):
            ok = np.allclose(ua.astype(float), e.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            ok = ua.tolist() == e.tolist()
    except Exception:
        ok = False
    return (True, "") if ok else (False, "La forma es correcta pero los valores no coinciden. Revisa el cálculo.")


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, np.ndarray):
        return _array_compare(u, e)
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    return False, f"Tu resultado es {u!r}, pero no es el valor esperado."


# ---------------------------------------------------------------- gráficos
def _plt():
    import matplotlib.pyplot as plt
    return plt


def _fig():
    plt = _plt()
    return plt.gcf() if plt.get_fignums() else None


def _ax(i=0):
    f = _fig()
    if f is None or len(f.axes) <= i:
        raise LookupError("no hay ningún gráfico dibujado")
    return f.axes[i]


def _legend(ax=None):
    ax = ax or _ax()
    lg = ax.get_legend()
    return [t.get_text() for t in lg.get_texts()] if lg else []


def _heights(ax=None):
    return [round(float(p.get_height()), 6) for p in (ax or _ax()).patches]


def _widths(ax=None):
    return [round(float(p.get_width()), 6) for p in (ax or _ax()).patches]


def _xticks(ax=None):
    return [t.get_text() for t in (ax or _ax()).get_xticklabels()]


def _hex(color):
    import matplotlib.colors as mcolors
    return mcolors.to_hex(color)


_HELPERS = {"_fig": _fig, "_ax": _ax, "_legend": _legend, "_heights": _heights,
            "_widths": _widths, "_xticks": _xticks, "_hex": _hex}


def pq_prepare():
    """Deja matplotlib limpio antes de ejecutar código (si está cargado)."""
    import sys
    if "matplotlib.pyplot" in sys.modules:
        plt = sys.modules["matplotlib.pyplot"]
        plt.close("all")
        plt.show = lambda *a, **k: None


def pq_check(ns, get_expected, opts):
    """Evalúa el ejercicio.

    ns = namespace del alumno. get_expected = función que devuelve el namespace de la
    solución (solo se ejecuta si hace falta, para no pisar los gráficos del alumno).
    """
    custom = opts.get("custom")
    if custom:
        if opts.get("plot") and _fig() is None:
            return False, "Todavía no hay ningún gráfico. ¡Dibuja algo! 🎨"
        env = dict(_HELPERS)
        env.update(ns)
        try:
            ok = bool(eval(custom, env))
        except Exception as err:
            return False, f"{opts.get('custom_msg', 'Todavía no es correcto.')} ({type(err).__name__}: {err})"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    exp_ns = get_expected() if callable(get_expected) else get_expected
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usa & (y), | (o) y pon cada condición entre paréntesis."),
    ("KeyError", "Revisa que el nombre de la columna esté bien escrito (mayúsculas, tildes, espacios)."),
    ("NameError", "Estás usando una variable que no existe. ¿Un error de tipeo?"),
    ("SyntaxError", "Hay un error de sintaxis: revisa paréntesis, comillas y que la línea `resultado = ...` esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este objeto. Revisa el nombre."),
    ("TypeError", "Algún tipo de dato no encaja (¿texto mezclado con números?)."),
    ("IndentationError", "Revisa la sangría (espacios al inicio de la línea)."),
    ("MergeError", "Revisa las columnas que usas para unir (on / left_on / right_on)."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import pandas as pd\nimport numpy as np\n\nclientes = pd.DataFrame({\n    "cliente_id": range(1, 13),\n    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo",\n               "Gabriela", "Hugo", "Inés", "Julián", "Karina", "Lucas"],\n    "ciudad": ["Madrid", "Buenos Aires", "CDMX", "Bogotá", "Madrid", "Lima",\n               "Buenos Aires", "CDMX", "Santiago", "Bogotá", "Madrid", "Lima"],\n    "pais": ["España", "Argentina", "México", "Colombia", "España", "Perú",\n             "Argentina", "México", "Chile", "Colombia", "España", "Perú"],\n    "edad": [34, 28, 45, 23, 51, 39, 30, 27, 42, 36, 25, 48],\n    "segmento": ["Premium", "Básico", "Premium", "Básico", "Premium", "Estándar",\n                 "Estándar", "Básico", "Premium", "Estándar", "Básico", "Estándar"],\n    "email": ["ana@mail.com", None, "carla@mail.com", "diego@mail.com", None, "facu@mail.com",\n              "gabi@mail.com", None, "ines@mail.com", "julian@mail.com", "kari@mail.com", "lucas@mail.com"],\n    "fecha_registro": pd.to_datetime([\n        "2023-01-15", "2023-02-03", "2023-02-20", "2023-03-11", "2023-04-02", "2023-05-19",\n        "2023-06-07", "2023-07-23", "2023-08-30", "2023-09-14", "2023-10-05", "2023-11-28"]),\n})\n\nproductos = pd.DataFrame({\n    "producto_id": range(1, 11),\n    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",\n                 "Webcam", "Silla gamer", "Escritorio", "Tablet", "Micrófono"],\n    "categoria": ["Computación", "Accesorios", "Accesorios", "Computación", "Audio",\n                  "Accesorios", "Muebles", "Muebles", "Computación", "Audio"],\n    "precio": [1200.0, 25.5, 45.0, 300.0, 80.0, 60.0, 250.0, 180.0, 450.0, 95.0],\n    "stock": [5, 150, 80, 20, 60, 0, 12, 7, 15, 0],\n})\n\n_estados = ["entregado", "enviado", "entregado", "cancelado", "entregado", "pendiente"]\npedidos = pd.DataFrame({\n    "pedido_id": range(1001, 1031),\n    "cliente_id": [(i * 7) % 11 + 1 for i in range(29)] + [99],\n    "producto_id": [(i * 3) % 10 + 1 for i in range(30)],\n    "cantidad": [(i % 4) + 1 for i in range(30)],\n    "fecha": pd.date_range("2024-01-03", periods=30, freq="5D"),\n    "estado": [_estados[i % 6] for i in range(30)],\n})\n\nempleados = pd.DataFrame({\n    "emp_id": range(1, 11),\n    "nombre": ["Marta", "Pablo", "Sofía", "Tomás", "Valeria",\n               "Andrés", "Lucía", "Martín", "Paula", "Ramiro"],\n    "depto": ["Dirección", "Ventas", "Ventas", "Ventas", "IT",\n              "IT", "IT", "Marketing", "Marketing", "Ventas"],\n    "salario": [9000, 4200, 3900, 4000, 5200, 4800, 6100, 3700, 4100, 3500],\n    "jefe_id": pd.array([None, 1, 2, 2, 1, 5, 5, 1, 8, 2], dtype="Int64"),\n    "fecha_ingreso": pd.to_datetime([\n        "2015-03-01", "2018-06-15", "2019-01-10", "2020-09-01", "2017-11-20",\n        "2021-02-01", "2016-08-08", "2022-04-18", "2019-07-07", "2023-01-09"]),\n})\n\n_i = np.arange(90)\n_f = pd.date_range("2024-01-01", periods=90, freq="D")\n_v = 1000 + 8 * _i + 250 * (_f.dayofweek >= 5) + ((_i * 37) % 11) * 20\n_v = np.where(np.isin(_i, [20, 55, 77]), _v + 1500, _v)\nventas_diarias = pd.DataFrame({\n    "fecha": _f,\n    "ventas": _v.astype(float),\n    "visitas": (300 + (_i * 13) % 97 + 2 * _i).astype(int),\n})\n\nsucio = pd.DataFrame({\n    "id": [1, 2, 2, 3, 4, 5, 6, 7],\n    "nombre": ["  ana ", "BRUNO", "BRUNO", "carla", None, "Diego  ", "elena", "facundo"],\n    "edad": ["34", "28", "28", "n/d", "23", "51", None, "39"],\n    "ciudad": ["madrid", "Buenos Aires", "Buenos Aires", " santiago", "bogotá", "Madrid ", "madrid", "LIMA"],\n    "monto": ["$1,200.50", "$300", "$300", "$45.00", None, "$80.5", "$99", "$1,000"],\n})\ndel _i, _f, _v, _estados\n\nimport matplotlib.pyplot as plt\n\nmeses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun"]\ningresos = [12.5, 14.1, 13.8, 16.2, 18.9, 21.3]   # miles\ngastos = [10.2, 11.0, 11.5, 12.1, 13.0, 13.4]     # miles\n'
_SOL_B64 = 'eyJwbHQwMSI6IHsic29sdXRpb24iOiAicGx0LnBsb3QobWVzZXMsIGluZ3Jlc29zKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLmxpbmVzKSA9PSAxIGFuZCBsaXN0KF9heCgpLmxpbmVzWzBdLmdldF95ZGF0YSgpKSA9PSBpbmdyZXNvcyIsICJjdXN0b21fbXNnIjogIkRlYmUgaGFiZXIgdW5hIGzDrW5lYSBjb24gbG9zIHZhbG9yZXMgZGUgYGluZ3Jlc29zYCBlbiBlbCBlamUgWS4ifSwgInhwIjogMTB9LCAicGx0MDIiOiB7InNvbHV0aW9uIjogInBsdC5wbG90KG1lc2VzLCBpbmdyZXNvcylcbnBsdC50aXRsZSgnSW5ncmVzb3MgcG9yIG1lcycpXG5wbHQueGxhYmVsKCdNZXMnKVxucGx0LnlsYWJlbCgnSW5ncmVzb3MgKG1pbGVzKScpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICJsZW4oX2F4KCkubGluZXMpID09IDEgYW5kIF9heCgpLmdldF90aXRsZSgpID09ICdJbmdyZXNvcyBwb3IgbWVzJyBhbmQgX2F4KCkuZ2V0X3hsYWJlbCgpID09ICdNZXMnIGFuZCBfYXgoKS5nZXRfeWxhYmVsKCkgPT0gJ0luZ3Jlc29zIChtaWxlcyknIiwgImN1c3RvbV9tc2ciOiAiUmV2aXNhIHF1ZSBlbCB0w610dWxvIHkgbGFzIGV0aXF1ZXRhcyBlc3TDqW4gZXNjcml0b3MgZXhhY3RhbWVudGUgY29tbyBzZSBwaWRlLiJ9LCAieHAiOiAxMH0sICJwbHQwMyI6IHsic29sdXRpb24iOiAicGx0LnBsb3QobWVzZXMsIGluZ3Jlc29zLCBsYWJlbD0nSW5ncmVzb3MnKVxucGx0LnBsb3QobWVzZXMsIGdhc3RvcywgbGFiZWw9J0dhc3RvcycpXG5wbHQubGVnZW5kKCkiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogImxlbihfYXgoKS5saW5lcykgPT0gMiBhbmQgc29ydGVkKF9sZWdlbmQoKSkgPT0gWydHYXN0b3MnLCAnSW5ncmVzb3MnXSIsICJjdXN0b21fbXNnIjogIkRlYmVuIHZlcnNlIDIgbMOtbmVhcyB5IHVuYSBsZXllbmRhIGNvbiAnSW5ncmVzb3MnIHkgJ0dhc3RvcycuIn0sICJ4cCI6IDIwfSwgInBsdDA0IjogeyJzb2x1dGlvbiI6ICJwbHQucGxvdChtZXNlcywgaW5ncmVzb3MsIGNvbG9yPSdncmVlbicsIG1hcmtlcj0nbycsIGxpbmVzdHlsZT0nLS0nKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLmxpbmVzKSA9PSAxIGFuZCBfaGV4KF9heCgpLmxpbmVzWzBdLmdldF9jb2xvcigpKSA9PSAnIzAwODAwMCcgYW5kIF9heCgpLmxpbmVzWzBdLmdldF9tYXJrZXIoKSA9PSAnbycgYW5kIF9heCgpLmxpbmVzWzBdLmdldF9saW5lc3R5bGUoKSA9PSAnLS0nIiwgImN1c3RvbV9tc2ciOiAiTGEgbMOtbmVhIGRlYmUgc2VyIHZlcmRlICgnZ3JlZW4nKSwgY29uIG1hcmNhZG9yICdvJyB5IGVzdGlsbyAnLS0nLiJ9LCAieHAiOiAyMH0sICJwbHQxMiI6IHsic29sdXRpb24iOiAieCA9IFswLCAxLCAyLCAzLCA0XVxueSA9IFsxLCA0LCA2LCA4LCAxMF1cbnBsdC5wbG90KHgsIHksICdncy0nKVxucGx0LmdyaWQoVHJ1ZSkiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogImxlbihfYXgoKS5saW5lcykgPT0gMSBhbmQgX2hleChfYXgoKS5saW5lc1swXS5nZXRfY29sb3IoKSkgPT0gJyMwMDgwMDAnIGFuZCBfYXgoKS5saW5lc1swXS5nZXRfbWFya2VyKCkgPT0gJ3MnIGFuZCBfYXgoKS5saW5lc1swXS5nZXRfbGluZXN0eWxlKCkgPT0gJy0nIGFuZCBfYXgoKS54YXhpcy5nZXRfZ3JpZGxpbmVzKClbMF0uZ2V0X3Zpc2libGUoKSIsICJjdXN0b21fbXNnIjogIkxhIGzDrW5lYSBkZWJlIHNlciB2ZXJkZSAoJ2cnKSwgY29uIGN1YWRyYWRvcyAoJ3MnKSB5IHPDs2xpZGEgKCctJykuIFkgbm8gb2x2aWRlcyBwbHQuZ3JpZChUcnVlKS4ifSwgInhwIjogMjB9LCAicGx0MTMiOiB7InNvbHV0aW9uIjogInggPSBbMCwgMSwgMiwgMywgNF1cbnkgPSBbMSwgNCwgNiwgOCwgMTBdXG5wbHQucGxvdCh4LCB5LCAnZ3MtJylcbnBsdC54bGltKC0xLCA2KVxucGx0LnlsaW0oLTIsIDE1KSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLmxpbmVzKSA9PSAxIGFuZCB0dXBsZShyb3VuZCh2LCA2KSBmb3IgdiBpbiBfYXgoKS5nZXRfeGxpbSgpKSA9PSAoLTEsIDYpIGFuZCB0dXBsZShyb3VuZCh2LCA2KSBmb3IgdiBpbiBfYXgoKS5nZXRfeWxpbSgpKSA9PSAoLTIsIDE1KSIsICJjdXN0b21fbXNnIjogIkVsIGVqZSBYIGRlYmUgaXIgZGUgLTEgYSA2IHkgZWwgZWplIFkgZGUgLTIgYSAxNS4ifSwgInhwIjogMjB9LCAicGx0MTQiOiB7InNvbHV0aW9uIjogIm51bWVyb3MgPSBbNCwgMiwgNywgNiwgM11cbnBsdC5wbG90KG51bWVyb3MpXG5wbHQuc2F2ZWZpZygnbWlfZ3JhZmljby5wbmcnLCBkcGk9MTUwKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLmxpbmVzKSA9PSAxIGFuZCAnc2F2ZWZpZycgaW4gX3BxX2NvZGUgYW5kICdtaV9ncmFmaWNvLnBuZycgaW4gX3BxX2NvZGUgYW5kIF9faW1wb3J0X18oJ29zJykucGF0aC5leGlzdHMoJ21pX2dyYWZpY28ucG5nJykiLCAiY3VzdG9tX21zZyI6ICJVc2EgcGx0LnNhdmVmaWcoJ21pX2dyYWZpY28ucG5nJywgZHBpPTE1MCkgZGVzcHXDqXMgZGUgZGlidWphci4ifSwgInhwIjogMTB9LCAicGx0MDUiOiB7InNvbHV0aW9uIjogInBsdC5iYXIocHJvZHVjdG9zWydwcm9kdWN0byddLCBwcm9kdWN0b3NbJ3N0b2NrJ10pXG5wbHQueHRpY2tzKHJvdGF0aW9uPTQ1KSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9heCgpLnBhdGNoZXMpID09IDEwIGFuZCBfaGVpZ2h0cygpID09IHByb2R1Y3Rvc1snc3RvY2snXS50b2xpc3QoKSIsICJjdXN0b21fbXNnIjogIkRlYmUgaGFiZXIgMTAgYmFycmFzIGNvbiBlbCBzdG9jayBkZSBjYWRhIHByb2R1Y3RvLCBlbiBlbCBvcmRlbiBkZSBsYSB0YWJsYS4ifSwgInhwIjogMTB9LCAicGx0MDYiOiB7InNvbHV0aW9uIjogInRvcCA9IHByb2R1Y3Rvcy5ubGFyZ2VzdCg1LCAncHJlY2lvJylcbnBsdC5iYXJoKHRvcFsncHJvZHVjdG8nXSwgdG9wWydwcmVjaW8nXSkiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogImxlbihfYXgoKS5wYXRjaGVzKSA9PSA1IGFuZCBzb3J0ZWQoX3dpZHRocygpKSA9PSBzb3J0ZWQocHJvZHVjdG9zLm5sYXJnZXN0KDUsICdwcmVjaW8nKVsncHJlY2lvJ10udG9saXN0KCkpIiwgImN1c3RvbV9tc2ciOiAiRGViZSBoYWJlciA1IGJhcnJhcyBob3Jpem9udGFsZXMgY29uIGxvcyBwcmVjaW9zIGRlIGxvcyA1IHByb2R1Y3RvcyBtw6FzIGNhcm9zLiJ9LCAieHAiOiAyMH0sICJwbHQwNyI6IHsic29sdXRpb24iOiAicGx0Lmhpc3QodmVudGFzX2RpYXJpYXNbJ3ZlbnRhcyddLCBiaW5zPTEyLCBlZGdlY29sb3I9J2JsYWNrJykiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogImxlbihfYXgoKS5wYXRjaGVzKSA9PSAxMiBhbmQgc3VtKF9oZWlnaHRzKCkpID09IDkwIiwgImN1c3RvbV9tc2ciOiAiRWwgaGlzdG9ncmFtYSBkZWJlIHRlbmVyIDEyIGJhcnJhcyBjb24gbG9zIDkwIGTDrWFzIGRlIHZlbnRhcy4ifSwgInhwIjogMjB9LCAicGx0MDgiOiB7InNvbHV0aW9uIjogInBsdC5zY2F0dGVyKHZlbnRhc19kaWFyaWFzWyd2aXNpdGFzJ10sIHZlbnRhc19kaWFyaWFzWyd2ZW50YXMnXSlcbnBsdC54bGFiZWwoJ1Zpc2l0YXMnKVxucGx0LnlsYWJlbCgnVmVudGFzJykiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogImxlbihfYXgoKS5jb2xsZWN0aW9ucykgPT0gMSBhbmQgbGVuKF9heCgpLmNvbGxlY3Rpb25zWzBdLmdldF9vZmZzZXRzKCkpID09IDkwIGFuZCBfYXgoKS5nZXRfeGxhYmVsKCkgPT0gJ1Zpc2l0YXMnIGFuZCBfYXgoKS5nZXRfeWxhYmVsKCkgPT0gJ1ZlbnRhcyciLCAiY3VzdG9tX21zZyI6ICJEZWJlIGhhYmVyIHVuIHNjYXR0ZXIgY29uIGxvcyA5MCBkw61hcyB5IGxhcyBldGlxdWV0YXMgJ1Zpc2l0YXMnIHkgJ1ZlbnRhcycuIn0sICJ4cCI6IDIwfSwgInBsdDA5IjogeyJzb2x1dGlvbiI6ICJjb250ZW8gPSBwcm9kdWN0b3NbJ2NhdGVnb3JpYSddLnZhbHVlX2NvdW50cygpXG5wbHQucGllKGNvbnRlbywgbGFiZWxzPWNvbnRlby5pbmRleCwgYXV0b3BjdD0nJTEuMGYlJScpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICJsZW4oX2F4KCkucGF0Y2hlcykgPT0gNCBhbmQgbGVuKF9heCgpLnRleHRzKSA9PSA4IiwgImN1c3RvbV9tc2ciOiAiTGEgdG9ydGEgZGViZSB0ZW5lciA0IHBvcmNpb25lcywgY29uIHN1IGV0aXF1ZXRhIHkgc3UgcG9yY2VudGFqZSBjYWRhIHVuYSAodXNhIGxhYmVscyB5IGF1dG9wY3QpLiJ9LCAieHAiOiAyMH0sICJwbHQxNSI6IHsic29sdXRpb24iOiAicm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDQyKVxuc2VyaWUxID0gcm5nLm5vcm1hbCgwLCAxLCA4MDApXG5zZXJpZTIgPSBybmcubm9ybWFsKDIsIDEsIDYwMClcbnNlcmllMyA9IHJuZy5ub3JtYWwoLTIsIDEsIDIwMClcbmZvciBzIGluIFtzZXJpZTEsIHNlcmllMiwgc2VyaWUzXTpcbiAgICBwbHQuaGlzdChzLCBiaW5zPTM1LCBoaXN0dHlwZT0nc3RlcCcsIGFscGhhPTAuNikiLCAiY2hlY2siOiB7InBsb3QiOiB0cnVlLCAiY3VzdG9tIjogImxlbihfYXgoKS5wYXRjaGVzKSA9PSAzIGFuZCBhbGwoYWJzKChwLmdldF9hbHBoYSgpIG9yIDApIC0gMC42KSA8IDFlLTkgZm9yIHAgaW4gX2F4KCkucGF0Y2hlcykgYW5kIG5vdCBhbnkocC5nZXRfZmlsbCgpIGZvciBwIGluIF9heCgpLnBhdGNoZXMpIiwgImN1c3RvbV9tc2ciOiAiRGViZW4gdmVyc2UgMyBoaXN0b2dyYW1hcyB0aXBvICdzdGVwJyAoc2luIHJlbGxlbm8pIGNvbiBhbHBoYT0wLjYuIn0sICJ4cCI6IDM1fSwgInBsdDE2IjogeyJzb2x1dGlvbiI6ICJ4MSA9IFswLCAxLCAyLCAzLCA0XVxueTEgPSBbMSwgNCwgNiwgOCwgMTBdXG54MiA9IFswLCAxLCAyLCAzLCA0XVxueTIgPSBbMCwgMSwgOCwgMjcsIDY0XVxucGx0LnNjYXR0ZXIoeDEsIHkxLCBjb2xvcj0nb3JhbmdlJywgbWFya2VyPSdzJywgbGFiZWw9J0Nvbmp1bnRvIEEnKVxucGx0LnNjYXR0ZXIoeDIsIHkyLCBjb2xvcj0ncHVycGxlJywgbWFya2VyPSdEJywgbGFiZWw9J0Nvbmp1bnRvIEInKVxucGx0LmxlZ2VuZCgpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICJsZW4oX2F4KCkuY29sbGVjdGlvbnMpID09IDIgYW5kIHNvcnRlZChfbGVnZW5kKCkpID09IFsnQ29uanVudG8gQScsICdDb25qdW50byBCJ10gYW5kIF9oZXgoX2F4KCkuY29sbGVjdGlvbnNbMF0uZ2V0X2ZhY2Vjb2xvcigpWzBdKSA9PSAnI2ZmYTUwMCcgYW5kIF9oZXgoX2F4KCkuY29sbGVjdGlvbnNbMV0uZ2V0X2ZhY2Vjb2xvcigpWzBdKSA9PSAnIzgwMDA4MCciLCAiY3VzdG9tX21zZyI6ICJEZWJlbiB2ZXJzZSAyIGdydXBvcyAobmFyYW5qYSB5IHDDunJwdXJhKSB5IHVuYSBsZXllbmRhIGNvbiAnQ29uanVudG8gQScgeSAnQ29uanVudG8gQicuIn0sICJ4cCI6IDIwfSwgInBsdDE3IjogeyJzb2x1dGlvbiI6ICJkZiA9IHBkLkRhdGFGcmFtZSh7XG4gICAgJ3Byb2R1Y3Rvcyc6IFsnTGFwdG9wcycsICdUYWJsZXRzJywgJ1RlbMOpZm9ub3MnLCAnQWNjZXNvcmlvcyddLFxuICAgICdjYW50aWRhZGVzJzogWzM1LCAzMCwgMjUsIDEwXVxufSlcbmNvbG9yZXMgPSBbJ2JsdWUnLCAnZ3JlZW4nLCAnZ29sZCcsICdsaWdodGNvcmFsJ11cbnBsdC5waWUoZGZbJ2NhbnRpZGFkZXMnXSwgbGFiZWxzPWRmWydwcm9kdWN0b3MnXSwgYXV0b3BjdD0nJTEuMmYlJScsIGNvbG9ycz1jb2xvcmVzLFxuICAgICAgICBzaGFkb3c9VHJ1ZSwgZXhwbG9kZT0oMCwgMCwgMC4xLCAwKSlcbnBsdC50aXRsZSgnSW52ZW50YXJpbyBkZSBwcm9kdWN0b3MgdGVjbm9sw7NnaWNvcycpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICJsZW4oW3AgZm9yIHAgaW4gX2F4KCkucGF0Y2hlcyBpZiB0eXBlKHApLl9fbmFtZV9fID09ICdXZWRnZSddKSA9PSA0IGFuZCAnMzUuMDAlJyBpbiBbdC5nZXRfdGV4dCgpIGZvciB0IGluIF9heCgpLnRleHRzXSBhbmQgJ1RlbMOpZm9ub3MnIGluIFt0LmdldF90ZXh0KCkgZm9yIHQgaW4gX2F4KCkudGV4dHNdIGFuZCBbX2hleChwLmdldF9mYWNlY29sb3IoKSkgZm9yIHAgaW4gX2F4KCkucGF0Y2hlcyBpZiB0eXBlKHApLl9fbmFtZV9fID09ICdXZWRnZSddID09IFsnIzAwMDBmZicsICcjMDA4MDAwJywgJyNmZmQ3MDAnLCAnI2YwODA4MCddIGFuZCB0dXBsZShyb3VuZCh2LCA2KSBmb3IgdiBpbiBbcCBmb3IgcCBpbiBfYXgoKS5wYXRjaGVzIGlmIHR5cGUocCkuX19uYW1lX18gPT0gJ1dlZGdlJ11bMl0uY2VudGVyKSAhPSAoMCwgMCkgYW5kIGxlbihfYXgoKS5wYXRjaGVzKSA+IDQgYW5kIF9heCgpLmdldF90aXRsZSgpID09ICdJbnZlbnRhcmlvIGRlIHByb2R1Y3RvcyB0ZWNub2zDs2dpY29zJyIsICJjdXN0b21fbXNnIjogIlJldmlzYTogbGFiZWxzLCBhdXRvcGN0PSclMS4yZiUlJywgY29sb3JzPWNvbG9yZXMsIHNoYWRvdz1UcnVlLCBleHBsb2RlPSgwLCAwLCAwLjEsIDApIHkgZWwgdMOtdHVsbyBleGFjdG8uIn0sICJ4cCI6IDM1fSwgInBsdDEwIjogeyJzb2x1dGlvbiI6ICJmaWcsIChheDEsIGF4MikgPSBwbHQuc3VicGxvdHMoMSwgMiwgZmlnc2l6ZT0oMTAsIDQpKVxuYXgxLnBsb3QobWVzZXMsIGluZ3Jlc29zKVxuYXgyLmJhcihtZXNlcywgZ2FzdG9zKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9maWcoKS5heGVzKSA9PSAyIGFuZCBsZW4oX2F4KDApLmxpbmVzKSA9PSAxIGFuZCBsZW4oX2F4KDEpLnBhdGNoZXMpID09IDYiLCAiY3VzdG9tX21zZyI6ICJJenF1aWVyZGE6IHVuYSBsw61uZWEgY29uIGluZ3Jlc29zLiBEZXJlY2hhOiA2IGJhcnJhcyBjb24gZ2FzdG9zLiJ9LCAieHAiOiAzNX0sICJwbHQxMSI6IHsic29sdXRpb24iOiAidmVudGFzX2RpYXJpYXMucGxvdCh4PSdmZWNoYScsIHk9J3ZlbnRhcycsIHRpdGxlPSdWZW50YXMgZGlhcmlhcycpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICJsZW4oX2F4KCkubGluZXMpID49IDEgYW5kIGxlbihfYXgoKS5saW5lc1swXS5nZXRfeWRhdGEoKSkgPT0gOTAgYW5kIF9heCgpLmdldF90aXRsZSgpID09ICdWZW50YXMgZGlhcmlhcyciLCAiY3VzdG9tX21zZyI6ICJEZWJlIHZlcnNlIHVuYSBsw61uZWEgY29uIGxvcyA5MCBkw61hcyB5IGVsIHTDrXR1bG8gJ1ZlbnRhcyBkaWFyaWFzJy4ifSwgInhwIjogMjB9LCAicGx0MTgiOiB7InNvbHV0aW9uIjogInggPSBbMCwgMSwgMiwgM11cbnkgPSBbMSwgMywgOSwgMjddXG5maWcsIChhcnJpYmEsIGFiYWpvKSA9IHBsdC5zdWJwbG90cygyLCAxKVxuYXJyaWJhLnBsb3QoeCwgeSlcbmFiYWpvLnNjYXR0ZXIoeCwgeSlcbnBsdC50aWdodF9sYXlvdXQoKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAibGVuKF9maWcoKS5heGVzKSA9PSAyIGFuZCBfYXgoMCkuZ2V0X3Bvc2l0aW9uKCkueTAgPiBfYXgoMSkuZ2V0X3Bvc2l0aW9uKCkueTAgYW5kIGxlbihfYXgoMCkubGluZXMpID09IDEgYW5kIGxlbihfYXgoMSkuY29sbGVjdGlvbnMpID09IDEiLCAiY3VzdG9tX21zZyI6ICJBcnJpYmEgZGViZSBoYWJlciB1bmEgbMOtbmVhIHkgYWJham8gdW4gc2NhdHRlciAodXNhIHBsdC5zdWJwbG90cygyLCAxKSkuIn0sICJ4cCI6IDIwfSwgInBsdDE5IjogeyJzb2x1dGlvbiI6ICJkYXRvcyA9IFszLCA3LCA0LCA5XVxuZmlndXJhLCBlamVzID0gcGx0LnN1YnBsb3RzKDMsIDIpXG5lamVzWzIsIDFdLnBsb3QoZGF0b3MpIiwgImNoZWNrIjogeyJwbG90IjogdHJ1ZSwgImN1c3RvbSI6ICJlamVzLnNoYXBlID09ICgzLCAyKSBhbmQgbGVuKF9maWcoKS5heGVzKSA9PSA2IGFuZCBsZW4oZWplc1syLCAxXS5saW5lcykgPT0gMSBhbmQgc3VtKGxlbihhLmxpbmVzKSBmb3IgYSBpbiBlamVzLmZsYXQpID09IDEiLCAiY3VzdG9tX21zZyI6ICJDcmVhIGBmaWd1cmEsIGVqZXMgPSBwbHQuc3VicGxvdHMoMywgMilgIHkgZGlidWphIHNvbG8gZW4gZWplc1syLCAxXS4ifSwgInhwIjogMjB9LCAicGx0MjAiOiB7InNvbHV0aW9uIjogIm1lc2VzID0gbGlzdChyYW5nZSgxLCAxMykpXG52ZW50YXMgPSBbMTIwLCAxMzUsIDExNSwgMTYwLCAxNzUsIDIxMCwgMTk4LCAyMjUsIDE5MCwgMjQwLCAyNjUsIDMxMF1cbnBsdC5maWd1cmUoZmFjZWNvbG9yPSdsaWdodGJsdWUnKVxucGx0LnBsb3QobWVzZXMsIHZlbnRhcylcbnBsdC5maWxsX2JldHdlZW4obWVzZXMsIHZlbnRhcywgYWxwaGE9MC4yKSIsICJjaGVjayI6IHsicGxvdCI6IHRydWUsICJjdXN0b20iOiAiX2hleChfZmlnKCkuZ2V0X2ZhY2Vjb2xvcigpKSA9PSAnI2FkZDhlNicgYW5kIGxlbihfYXgoKS5saW5lcykgPT0gMSBhbmQgYW55KCdQb2x5JyBpbiB0eXBlKGMpLl9fbmFtZV9fIGFuZCBhYnMoKGMuZ2V0X2FscGhhKCkgb3IgMCkgLSAwLjIpIDwgMWUtOSBmb3IgYyBpbiBfYXgoKS5jb2xsZWN0aW9ucykiLCAiY3VzdG9tX21zZyI6ICJMYSBmaWd1cmEgZGViZSB0ZW5lciBmb25kbyAnbGlnaHRibHVlJywgdW5hIGzDrW5lYSB5IGVsIMOhcmVhIHJlbGxlbmEgY29uIGZpbGxfYmV0d2VlbihhbHBoYT0wLjIpLiJ9LCAieHAiOiAzNX19'
_TABLAS = ['meses', 'ingresos', 'gastos', 'productos', 'ventas_diarias']

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todas las tablas limpias (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())
    pq_prepare()

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    def esperado():
        exp_ns = {}
        exec(_SETUP, exp_ns)
        exec(info["solution"], exp_ns)
        return exp_ns
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, esperado, info["check"])
    if ok:
        from IPython.display import HTML, display
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">🎉 ¡Correcto! +' + str(info["xp"]) + ' XP · Sigue así 🐼</div>'))
    else:
        from IPython.display import HTML, display
        import html as _h
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">❌ Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_solucion(ej):
    print(_SOL[ej]["solution"])

reiniciar_datos()
print("✅ Listo." + (" Datos cargados: " + ", ".join(_TABLAS) if _TABLAS else " Ya puedes empezar."))


---
# 📌 Primeros gráficos

## 1. Tu primer gráfico
`Básico` · **+10 XP**

**Matplotlib** es la librería de gráficos más usada de Python y la base de otras como seaborn o el `.plot()` de pandas. Su módulo principal se importa así:

```python
import matplotlib.pyplot as plt
```

El gráfico más básico es el de **líneas**: `plt.plot(x, y)` une los puntos con una línea. Es ideal para ver cómo **evoluciona** algo en el tiempo.

```python
plt.plot([1, 2, 3], [10, 30, 20])
```

En este módulo tienes cargadas las listas `meses`, `ingresos` y `gastos` (en miles), además de las tablas del curso.

> 💡 En Colab/Jupyter el gráfico aparece solo al final de la celda. En un script de Python hace falta `plt.show()`.

### 🎯 Tu misión
Dibuja un gráfico de líneas con `meses` en el eje X e `ingresos` en el eje Y.

<details><summary>💡 Pista</summary>

`plt.plot(meses, ingresos)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

# plt ya está importado


comprobar("plt01")

## 2. Título y etiquetas
`Básico` · **+10 XP**

Un gráfico sin título ni etiquetas **no comunica**: quien lo mire no sabrá qué representa. Siempre añade:

```python
plt.title("Ventas 2024")
plt.xlabel("Mes")
plt.ylabel("Unidades vendidas")
```

> 💼 **En el trabajo:** un gráfico en una presentación debe entenderse **en 5 segundos** sin que nadie lo explique. Título claro, ejes con nombre y unidades.

### 🎯 Tu misión
Dibuja `ingresos` por mes y añade el título **"Ingresos por mes"**, la etiqueta X **"Mes"** y la etiqueta Y **"Ingresos (miles)"** (escríbelos exactamente así).

<details><summary>💡 Pista</summary>

`plt.title('...'), plt.xlabel('...'), plt.ylabel('...')`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

plt.plot(meses, ingresos)
# añade título y etiquetas


comprobar("plt02")

## 3. Comparar con leyenda
`Intermedio` · **+20 XP**

Puedes dibujar **varias líneas** en el mismo gráfico llamando a `plt.plot` varias veces. Para distinguirlas, pon a cada una un `label` y llama a `plt.legend()`:

```python
plt.plot(x, ventas_2023, label="2023")
plt.plot(x, ventas_2024, label="2024")
plt.legend()
```

Matplotlib asigna un color distinto a cada línea automáticamente.

### 🎯 Tu misión
Dibuja `ingresos` y `gastos` por mes en el mismo gráfico, con las etiquetas **"Ingresos"** y **"Gastos"**, y muestra la leyenda.

<details><summary>💡 Pista</summary>

`Dos plt.plot(..., label='...') y luego plt.legend()`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios



comprobar("plt03")

## 4. Dale estilo
`Intermedio` · **+20 XP**

`plt.plot` acepta parámetros de estilo:

| Parámetro | Ejemplos |
|---|---|
| `color` | `"red"`, `"green"`, `"#ff5733"` |
| `marker` | `"o"` (círculo), `"s"` (cuadrado), `"^"` (triángulo) |
| `linestyle` | `"-"` (sólida), `"--"` (guiones), `":"` (puntos) |
| `linewidth` | `2`, `3`… |

```python
plt.plot(x, y, color="purple", marker="s", linestyle=":")
```

> 💡 Los marcadores ayudan cuando hay pocos puntos: así se ve dónde está cada dato real.

### 🎯 Tu misión
Dibuja `ingresos` por mes en color **"green"**, con marcador **"o"** y línea de guiones **"--"**.

<details><summary>💡 Pista</summary>

`color='green', marker='o', linestyle='--'`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

plt.plot(meses, ingresos)

comprobar("plt04")

## 5. Formato abreviado y cuadrícula
`Intermedio` · **+20 XP**

En lugar de escribir `color=`, `marker=` y `linestyle=` por separado, `plot` acepta un **formato abreviado** como tercer argumento: un texto corto que combina **color + marcador + línea**.

| Código | Significado |
|---|---|
| `'r--'` | rojo, línea discontinua |
| `'bo'` | azul, solo círculos (sin línea) |
| `'g^-'` | verde, triángulos y línea sólida |
| `'ks:'` | negro (*k*), cuadrados y línea punteada |

Colores: `r` rojo, `g` verde, `b` azul, `k` negro, `m` magenta, `c` cian, `y` amarillo.

La **cuadrícula** ayuda a leer los valores. `alpha` la hace más suave:

```python
plt.plot(x, y, 'bo-')
plt.grid(True, alpha=0.3)
```

### 🎯 Tu misión
Dibuja `y` en función de `x` con **línea verde continua y marcadores cuadrados** usando el formato abreviado, y activa la **cuadrícula**.

<details><summary>💡 Pista</summary>

`plt.plot(x, y, 'gs-')  y  plt.grid(True)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

x = [0, 1, 2, 3, 4]
y = [1, 4, 6, 8, 10]
plt.plot(x, y)

comprobar("plt12")

## 6. Límites de los ejes
`Intermedio` · **+20 XP**

Matplotlib elige los límites de los ejes automáticamente, pero a veces conviene fijarlos: para comparar varios gráficos con la misma escala, o para enfocar una zona.

```python
plt.xlim(0, 10)              # eje X de 0 a 10
plt.ylim(-5, 100)            # eje Y de -5 a 100
plt.axis([0, 10, -5, 100])   # lo mismo en una línea: [xmin, xmax, ymin, ymax]
```

> ⚠️ Cuidado con cortar el eje Y para exagerar diferencias: un gráfico de barras que no empieza en 0 puede engañar a quien lo lee.

### 🎯 Tu misión
Dibuja `y` en función de `x` y fija el **eje X de -1 a 6** y el **eje Y de -2 a 15**.

<details><summary>💡 Pista</summary>

`plt.xlim(-1, 6)  y  plt.ylim(-2, 15)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

x = [0, 1, 2, 3, 4]
y = [1, 4, 6, 8, 10]
plt.plot(x, y, 'gs-')

comprobar("plt13")

## 7. Guardar tu gráfico
`Básico` · **+10 XP**

`plt.savefig()` guarda el gráfico en un archivo para usarlo en un informe, una presentación o una web:

```python
plt.plot(numeros)
plt.savefig("mi_grafico.png", dpi=300, bbox_inches="tight")
```

- La **extensión** define el formato: `.png` (imagen), `.pdf` o `.svg` (vectorial: no pierde calidad al agrandarse).
- `dpi=300` → alta resolución, ideal para imprimir.
- `bbox_inches="tight"` → recorta los márgenes blancos sobrantes.

> ⚠️ Llama a `savefig` **antes** de `plt.show()`: después de mostrarse, la figura se vacía y guardarías una imagen en blanco.

### 🎯 Tu misión
Dibuja la línea de `numeros` y guárdala como **`mi_grafico.png`** con `dpi=150`.

<details><summary>💡 Pista</summary>

`plt.savefig('mi_grafico.png', dpi=150)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

numeros = [4, 2, 7, 6, 3]
plt.plot(numeros)

comprobar("plt14")

---
# 📌 Tipos de gráfico

## 8. Gráfico de barras
`Básico` · **+10 XP**

Las **barras** sirven para **comparar cantidades entre categorías** (productos, países, vendedores…):

```python
plt.bar(categorias, valores)
```

Si los nombres son largos y se superponen, gíralos:

```python
plt.xticks(rotation=45)
```

> 💡 **¿Líneas o barras?** Líneas para tendencias en el tiempo; barras para comparar categorías.

### 🎯 Tu misión
Dibuja un gráfico de barras con el `stock` de cada `producto` de la tabla `productos`.

<details><summary>💡 Pista</summary>

`plt.bar(productos['producto'], productos['stock'])`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios



comprobar("plt05")

## 9. Barras horizontales: un top 5
`Intermedio` · **+20 XP**

`plt.barh(categorias, valores)` dibuja **barras horizontales**. Son ideales para los rankings, porque los nombres se leen cómodamente.

Lo habitual es preparar los datos primero con pandas y después graficar:

```python
top = df.nlargest(5, "ventas")
plt.barh(top["vendedor"], top["ventas"])
```

> 💼 **En el trabajo:** "Top 10 productos", "Top 5 clientes"… los gráficos de ranking están en todos los dashboards.

### 🎯 Tu misión
Obtén los **5 productos más caros** con `nlargest` y dibújalos con barras horizontales (`producto` y `precio`).

<details><summary>💡 Pista</summary>

`top = productos.nlargest(5, 'precio'); plt.barh(top['producto'], top['precio'])`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

top = 


comprobar("plt06")

## 10. Histograma
`Intermedio` · **+20 XP**

Un **histograma** muestra la **distribución** de una variable numérica: divide el rango en intervalos (*bins*) y cuenta cuántos valores caen en cada uno.

```python
plt.hist(datos, bins=20, edgecolor="black")
```

Con él respondes preguntas como: ¿los valores se concentran en el centro? ¿hay valores extremos? ¿la distribución es simétrica?

> ⚠️ El número de `bins` cambia mucho la lectura: con pocos se pierde detalle y con demasiados aparece ruido. Prueba varios.

### 🎯 Tu misión
Dibuja un histograma de la columna `ventas` de `ventas_diarias` con **12 bins**.

<details><summary>💡 Pista</summary>

`plt.hist(ventas_diarias['ventas'], bins=12)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios



comprobar("plt07")

## 11. Dispersión: ¿hay relación?
`Intermedio` · **+20 XP**

El gráfico de **dispersión** (*scatter*) pone un punto por cada observación, con una variable en cada eje. Sirve para ver si dos variables están **relacionadas**:

- los puntos suben juntos: relación **positiva**;
- uno sube y el otro baja: relación **negativa**;
- nube sin forma: **no hay relación**.

```python
plt.scatter(df["publicidad"], df["ventas"], alpha=0.6)
```

`alpha` añade transparencia, que viene bien cuando hay muchos puntos superpuestos.

### 🎯 Tu misión
Dibuja un gráfico de dispersión de `ventas_diarias` con `visitas` en X y `ventas` en Y. Pon las etiquetas de eje **"Visitas"** y **"Ventas"**.

<details><summary>💡 Pista</summary>

`plt.scatter(x, y) + plt.xlabel / plt.ylabel`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios



comprobar("plt08")

## 12. Gráfico de torta
`Intermedio` · **+20 XP**

El gráfico de **torta** (*pie*) muestra **proporciones de un total**. `autopct` escribe el porcentaje dentro de cada porción:

```python
plt.pie(valores, labels=nombres, autopct="%1.0f%%")
```

El formato `"%1.0f%%"` significa: número sin decimales seguido del símbolo `%`.

> ⚠️ Úsalo solo con **pocas categorías** (hasta unas 5). Con más, un gráfico de barras se lee mucho mejor.

### 🎯 Tu misión
Cuenta cuántos productos hay por `categoria` (con `value_counts`) y dibuja una torta con las etiquetas de cada categoría y los porcentajes (`autopct`).

<details><summary>💡 Pista</summary>

`plt.pie(conteo, labels=conteo.index, autopct='%1.0f%%')`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

conteo = productos['categoria'].value_counts()


comprobar("plt09")

## 13. Histogramas superpuestos
`Avanzado` · **+35 XP**

Para **comparar distribuciones** se pueden dibujar varios histogramas en el mismo gráfico. El truco es que no se tapen entre sí:

- `histtype="step"` dibuja solo el **contorno** de cada histograma.
- `alpha` (de 0 a 1) da **transparencia**.
- `bins` define cuántos intervalos usar. Con más bins ves más detalle, pero también más ruido.

```python
plt.hist(grupo_a, bins=30, histtype="step", alpha=0.7, label="A")
plt.hist(grupo_b, bins=30, histtype="step", alpha=0.7, label="B")
plt.legend()
```

Otros parámetros útiles de `plt.hist`: `density=True` (el eje Y muestra densidad en vez de conteo) y `color` / `edgecolor` (relleno y contorno).

### 🎯 Tu misión
Dibuja los **tres** histogramas (`serie1`, `serie2`, `serie3`) en el mismo gráfico, cada uno con `bins=35`, `histtype='step'` y `alpha=0.6`.

<details><summary>💡 Pista</summary>

`plt.hist(serie1, bins=35, histtype='step', alpha=0.6)  (y lo mismo con serie2 y serie3)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

rng = np.random.default_rng(42)
serie1 = rng.normal(0, 1, 800)
serie2 = rng.normal(2, 1, 600)
serie3 = rng.normal(-2, 1, 200)


comprobar("plt15")

## 14. Scatter con dos grupos
`Intermedio` · **+20 XP**

`plt.scatter` permite personalizar cada grupo de puntos:

| Parámetro | Qué controla | Ejemplo |
|---|---|---|
| `color` o `c` | color de los puntos | `"orange"` |
| `s` | tamaño | `80` |
| `marker` | forma | `"s"` cuadrado, `"D"` diamante, `"^"` triángulo |
| `label` | nombre en la leyenda | `"Grupo A"` |

```python
plt.scatter(x1, y1, color="orange", marker="s", label="Grupo A")
plt.scatter(x2, y2, color="purple", marker="D", label="Grupo B")
plt.legend()
```

> 💡 `plt.plot(x, y, 'o')` también dibuja puntos y es más rápido, pero **todos iguales**. `scatter` permite un color y un tamaño distinto **para cada punto** (útil para mostrar 4 variables a la vez).

### 🎯 Tu misión
Dibuja dos grupos de puntos: el **Conjunto A** (`x1`, `y1`) en color `'orange'` con marcador `'s'`, y el **Conjunto B** (`x2`, `y2`) en color `'purple'` con marcador `'D'`. Agrega la leyenda.

<details><summary>💡 Pista</summary>

`plt.scatter(x1, y1, color='orange', marker='s', label='Conjunto A')  ...  plt.legend()`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

x1 = [0, 1, 2, 3, 4]
y1 = [1, 4, 6, 8, 10]
x2 = [0, 1, 2, 3, 4]
y2 = [0, 1, 8, 27, 64]


comprobar("plt16")

## 15. Torta profesional
`Avanzado` · **+35 XP**

`plt.pie` tiene varios parámetros para que la torta se lea mejor:

| Parámetro | Qué hace |
|---|---|
| `labels` | nombre de cada porción |
| `autopct` | formato del porcentaje: `'%1.1f%%'` (1 decimal), `'%1.2f%%'` (2 decimales) |
| `colors` | lista con un color por porción |
| `explode` | separa porciones del centro: `(0, 0, 0.1, 0)` separa la tercera |
| `shadow` | `True` agrega sombra |
| `startangle` | ángulo donde empieza la primera porción (90 = arriba) |

```python
plt.pie(valores, labels=nombres, autopct="%1.1f%%", explode=(0, 0.1, 0), startangle=90)
plt.axis("equal")   # que sea un círculo perfecto
```

> ⚠️ Úsala con **pocas categorías** (6 como máximo) y diferencias claras. Si no, un gráfico de barras se lee mejor.

### 🎯 Tu misión
Con `df`, dibuja una torta con las etiquetas de cada producto, porcentajes con **2 decimales** (`'%1.2f%%'`), los `colores` dados, **sombra**, la porción de **Teléfonos separada** (`explode`) y el título **'Inventario de productos tecnológicos'**.

<details><summary>💡 Pista</summary>

`plt.pie(df['cantidades'], labels=df['productos'], autopct='%1.2f%%', colors=colores, shadow=True, explode=(0, 0, 0.1, 0))`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

df = pd.DataFrame({
    'productos': ['Laptops', 'Tablets', 'Teléfonos', 'Accesorios'],
    'cantidades': [35, 30, 25, 10]
})
colores = ['blue', 'green', 'gold', 'lightcoral']
plt.pie(df['cantidades'])

comprobar("plt17")

---
# 📌 Composición y pandas

## 16. Varios gráficos juntos
`Avanzado` · **+35 XP**

Hasta ahora usaste la interfaz rápida (`plt.plot`). Para gráficos más complejos se usa la **interfaz orientada a objetos**, con una **figura** (`fig`) que contiene varios **ejes** (`ax`):

```python
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))   # 1 fila, 2 columnas
ax1.plot(x, y)
ax1.set_title("Evolución")
ax2.bar(cats, vals)
ax2.set_title("Comparación")
```

Fíjate en que con ejes se usa `ax.set_title()` en lugar de `plt.title()`.

> 💼 **En el trabajo:** los informes suelen tener varios gráficos lado a lado. `plt.subplots` es la forma profesional de montarlos.

### 🎯 Tu misión
Crea una figura con **2 gráficos lado a lado**: a la izquierda, una línea de `ingresos` por mes; a la derecha, barras de `gastos` por mes.

<details><summary>💡 Pista</summary>

`ax1.plot(meses, ingresos) y ax2.bar(meses, gastos)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))


comprobar("plt10")

## 17. Graficar directo desde pandas
`Intermedio` · **+20 XP**

Los DataFrames tienen el método **`.plot()`**, que usa matplotlib por debajo. Es la forma más rápida de graficar datos que ya están en una tabla:

```python
df.plot(x="fecha", y="ventas")                  # líneas
df.plot(x="producto", y="precio", kind="bar")   # barras
df["edad"].plot(kind="hist", bins=10)           # histograma
```

También acepta `title=`, `figsize=`, `color=`…

> 💼 **En el trabajo:** para explorar datos, `df.plot()` es lo más cómodo. Para un gráfico final pulido, matplotlib o seaborn.

### 🎯 Tu misión
Con `ventas_diarias.plot`, dibuja las `ventas` a lo largo de la `fecha` con el título **"Ventas diarias"**.

<details><summary>💡 Pista</summary>

`ventas_diarias.plot(x='fecha', y='ventas', title='Ventas diarias')`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios



comprobar("plt11")

## 18. Uno encima del otro
`Intermedio` · **+20 XP**

`plt.subplots(filas, columnas)` crea una **figura** (el lienzo completo) con varios **ejes** (cada gráfico). La analogía: la figura es una pared y cada eje es un cuadro colgado en ella.

```python
fig, (arriba, abajo) = plt.subplots(2, 1, figsize=(6, 6))   # 2 filas, 1 columna
arriba.plot(x, y)
abajo.scatter(x, y)
plt.tight_layout()   # ajusta los márgenes para que nada se superponga
```

Con una cuadrícula de varias filas **y** columnas, los ejes vienen en un array y se accede con `[fila, columna]`:

```python
fig, axes = plt.subplots(2, 2)
axes[0, 0].plot(...)   # arriba a la izquierda
axes[1, 1].bar(...)    # abajo a la derecha
```

> 💡 `sharex=True` hace que los gráficos compartan el eje X: ideal para comparar series en el mismo período.

### 🎯 Tu misión
Crea una figura con **2 gráficos, uno encima del otro**: arriba una **línea** y abajo un **scatter**, los dos con los datos `x` e `y`.

<details><summary>💡 Pista</summary>

`arriba.plot(x, y)  y  abajo.scatter(x, y)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

x = [0, 1, 2, 3]
y = [1, 3, 9, 27]
fig, (arriba, abajo) = plt.subplots(2, 1)


comprobar("plt18")

## 19. Cuadrícula de 3 × 2
`Intermedio` · **+20 XP**

Cuando necesitas muchos gráficos (un **dashboard**), `plt.subplots` arma la cuadrícula completa de una vez. El segundo valor que devuelve es un **array de NumPy** con todos los ejes:

```python
figura, ejes = plt.subplots(3, 2, figsize=(10, 9))
ejes.shape          # (3, 2)
ejes[2, 1]          # el gráfico de la tercera fila, segunda columna
```

`figsize=(ancho, alto)` se mide en pulgadas. Y `figura.suptitle("...")` pone un título general arriba de todo.

> 💼 **En el trabajo:** un panel de 3 × 2 con las métricas clave (ventas, clientes, stock…) es un formato clásico de reporte mensual.

### 🎯 Tu misión
Crea una cuadrícula de **3 filas por 2 columnas** guardando la figura en `figura` y los ejes en `ejes`. Dibuja una línea con `datos` en el gráfico de **abajo a la derecha**.

<details><summary>💡 Pista</summary>

`figura, ejes = plt.subplots(3, 2)  y  ejes[2, 1].plot(datos)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

datos = [3, 7, 4, 9]


comprobar("plt19")

## 20. Fondo, área y valores
`Avanzado` · **+35 XP**

Tres detalles que hacen que un gráfico se vea profesional:

**1. Color de fondo de la figura**, con `facecolor`:

```python
plt.figure(figsize=(8, 4), facecolor="lightblue")
```

**2. Área bajo la curva**, con `fill_between`. Rellena entre la línea y el eje, y resalta la tendencia:

```python
plt.plot(meses, ventas, lw=2)
plt.fill_between(meses, ventas, alpha=0.15)
```

**3. Valores sobre las barras**, con `bar_label` (Matplotlib 3.4 o superior):

```python
barras = plt.bar(cats, vals)
plt.bar_label(barras)
```

### 🎯 Tu misión
Crea una figura con fondo **`'lightblue'`**, dibuja la línea de `ventas` por mes y rellena el **área bajo la curva** con `fill_between` y `alpha=0.2`.

<details><summary>💡 Pista</summary>

`plt.figure(facecolor='lightblue')  ->  plt.plot(...)  ->  plt.fill_between(meses, ventas, alpha=0.2)`
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

meses = list(range(1, 13))
ventas = [120, 135, 115, 160, 175, 210, 198, 225, 190, 240, 265, 310]
plt.plot(meses, ventas)

comprobar("plt20")

---
# 🏁 ¡Módulo completado!
Vuelve a [PandasQuest](https://rosalesluciano.github.io/pandas-quest/) para sumar tus XP y seguir con el siguiente módulo. 🚀